In [2]:
import os, sys
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import time
import shutil
import collections
from pathlib import Path

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [5]:
from py.utils import verifyDir, verifyFile, verifyType

In [6]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/home/vdela/UrbaNet-main/data/', '/home/vdela/UrbaNet-main/models/')

In [7]:
QSCORE_PATH=f"{cfg.DATA_PATH}pp2/{cfg.SCORING_METHOD}/{cfg.PLACE_LEVEL}/"
IMAGES_PATH = f"{cfg.DATA_PATH}pp2/images/"
MODEL_PATH = f"{cfg.MODEL_PATH}pp2/cnn/{cfg.SCORING_METHOD}/{cfg.PLACE_LEVEL}/"
EXPLAIN_PATH = f"{cfg.MODEL_PATH}pp2/explanations/"

In [8]:
verifyDir(MODEL_PATH)
verifyDir(EXPLAIN_PATH)

In [9]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_type = torch.float32 if device.type == "cuda" else torch.float16
device, torch_type

(device(type='cuda'), torch.float32)

In [10]:
NUM_CLASSES = 1 if "reg" in cfg.ML_TASK else 2

In [11]:
%%time
data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
data_df["image_path"] = f"{IMAGES_PATH}" + data_df["image_path"]
data_df["image_id"] = data_df["image_id"].apply(str)
data_df.sort_values(by=[cfg.PERCEPTION_METRIC], ascending=False, inplace=True)
data_df

CPU times: user 179 ms, sys: 61.6 ms, total: 241 ms
Wall time: 255 ms


,image_id,lat,long,city,country,continent,safety,beautiful,wealthy,lively,boring,depressing,image_path
98091,51414746fdc9f04926006a00,44.961377,-93.271491,Minneapolis,USA,North America,8.780423,7.500000,7.777778,6.278770,5.000000,3.333333,/home/vdela/UrbaNet-main/data/pp2/images/Minne...
42815,513d677cfdc9f035870040af,42.370774,-71.126977,Boston,USA,North America,8.583389,5.333333,6.055556,5.029020,3.333333,9.166667,/home/vdela/UrbaNet-main/data/pp2/images/Bosto...
50565,513d7c38fdc9f03587006e0f,33.805683,-84.293833,Atlanta,USA,North America,8.505291,6.342593,8.452381,2.291667,0.000000,3.333333,/home/vdela/UrbaNet-main/data/pp2/images/Atlan...
16840,50f5642cfdc9f065f00060f0,47.583294,-122.287884,Seattle,USA,North America,8.478006,7.166667,7.500000,5.538597,3.333333,2.777778,/home/vdela/UrbaNet-main/data/pp2/images/Seatt...
32864,513cc19efdc9f035870014bd,29.751635,-95.466232,Houston,USA,North America,8.440027,7.592593,6.653439,4.321429,2.708333,1.666667,/home/vdela/UrbaNet-main/data/pp2/images/Houst...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1768,50f42c0bfdc9f065f0001786,52.242431,20.898414,Warsaw,Poland,Europe,0.392157,5.899471,2.833333,4.872958,5.740741,3.333333,/home/vdela/UrbaNet-main/data/pp2/images/Warsa...
67419,513e6bb5fdc9f0358700c081,35.646141,139.812366,Tokyo,Japan,Asia,0.277778,3.333333,3.333333,4.538332,5.277778,7.438272,/home/vdela/UrbaNet-main/data/pp2/images/Tokyo...
15585,50f562dafdc9f065f0005ae6,1.292158,103.808340,Singapore,Singapore,Asia,0.256410,4.305556,3.333333,6.666667,5.333333,4.444444,/home/vdela/UrbaNet-main/data/pp2/images/Singa...
68591,513e6f20fdc9f0358700c51f,35.743776,139.773877,Tokyo,Japan,Asia,0.196078,5.092593,3.080808,6.603175,3.055556,2.500000,/home/vdela/UrbaNet-main/data/pp2/images/Tokyo...


In [12]:
rio_df = data_df[data_df["city"] == "Rio De Janeiro"].copy()
print(f"Imágenes en Rio de Janeiro: {len(rio_df)}")
rio_df.head(3)


Imágenes en Rio de Janeiro: 3644


,image_id,lat,long,city,country,continent,safety,beautiful,wealthy,lively,boring,depressing,image_path
22133,50f5eaecfdc9f065f0007e32,-22.901318,-43.178843,Rio De Janeiro,Brasil,South America,8.074111,7.222222,6.521164,5.259259,2.444444,5.000000,/home/vdela/UrbaNet-main/data/pp2/images/Rio D...
23068,50f5eb4bfdc9f065f00081e0,-22.924839,-43.387912,Rio De Janeiro,Brasil,South America,7.990805,3.015873,3.333333,3.767806,2.500000,2.261905,/home/vdela/UrbaNet-main/data/pp2/images/Rio D...
23210,50f5eb65fdc9f065f000826f,-22.829954,-43.374070,Rio De Janeiro,Brasil,South America,7.920455,3.740079,3.117284,3.333333,2.222222,5.000000,/home/vdela/UrbaNet-main/data/pp2/images/Rio D...


In [13]:
### SOLO LAS IAMGENES ESCOGIDAS EN EL EXPERIMENTO
image_ids = ['50f5ea5bfdc9f065f0007ab6',
 '50f5ea5bfdc9f065f0007ab9',
 '50f5ea5cfdc9f065f0007abf',
 '50f5ea5cfdc9f065f0007ac5',
 '50f5ea5dfdc9f065f0007ada',
 '50f5ea5ffdc9f065f0007b06',
 '50f5ea60fdc9f065f0007b19',
 '50f5ea61fdc9f065f0007b1d',
 '50f5ea61fdc9f065f0007b1f',
 '50f5ea61fdc9f065f0007b22',
 '50f5ea61fdc9f065f0007b27',
 '50f5eaa5fdc9f065f0007b7b',
 '50f5eaa5fdc9f065f0007b7c',
 '50f5eaa5fdc9f065f0007b81',
 '50f5eaa6fdc9f065f0007b8f',
 '50f5eaa6fdc9f065f0007b94',
 '50f5eaa7fdc9f065f0007ba0',
 '50f5eaa7fdc9f065f0007ba4',
 '50f5eaa9fdc9f065f0007bc7',
 '50f5eac6fdc9f065f0007ca1',
 '50f5eac6fdc9f065f0007cab',
 '50f5eac7fdc9f065f0007cb2',
 '50f5eacafdc9f065f0007cfd',
 '50f5eacbfdc9f065f0007d01',
 '50f5ead3fdc9f065f0007d9f',
 '50f5ead3fdc9f065f0007da4',
 '50f5ead3fdc9f065f0007da9',
 '50f5ead4fdc9f065f0007daa',
 '50f5ead4fdc9f065f0007db2',
 '50f5ead5fdc9f065f0007dbe',
 '50f5ead5fdc9f065f0007dbf',
 '50f5ead5fdc9f065f0007dc2',
 '50f5ead5fdc9f065f0007dc7',
 '50f5ead5fdc9f065f0007dca',
 '50f5ead5fdc9f065f0007dcb',
 '50f5ead6fdc9f065f0007dcd',
 '50f5ead6fdc9f065f0007dcf',
 '50f5ead6fdc9f065f0007dd8',
 '50f5eae9fdc9f065f0007dea',
 '50f5eaeafdc9f065f0007e08',
 '50f5eaecfdc9f065f0007e2c',
 '50f5eaecfdc9f065f0007e2d',
 '50f5eaecfdc9f065f0007e2f',
 '50f5eaecfdc9f065f0007e30',
 '50f5eaecfdc9f065f0007e36',
 '50f5eaedfdc9f065f0007e40',
 '50f5eaedfdc9f065f0007e46',
 '50f5eaedfdc9f065f0007e4b',
 '50f5eaeefdc9f065f0007e50',
 '50f5eaeefdc9f065f0007e55',
 '50f5eaeefdc9f065f0007e58',
 '50f5eaeefdc9f065f0007e59',
 '50f5eaeefdc9f065f0007e5a',
 '50f5eaeefdc9f065f0007e5e',
 '50f5eaeffdc9f065f0007e5f',
 '50f5eaeffdc9f065f0007e68',
 '50f5eaf0fdc9f065f0007e7e',
 '50f5eaf0fdc9f065f0007e81',
 '50f5eaf0fdc9f065f0007e82',
 '50f5eaf1fdc9f065f0007e84',
 '50f5eaf1fdc9f065f0007e86',
 '50f5eaf1fdc9f065f0007e87',
 '50f5eaf1fdc9f065f0007e88',
 '50f5eaf1fdc9f065f0007e8b',
 '50f5eaf1fdc9f065f0007e8c',
 '50f5eaf1fdc9f065f0007e8f',
 '50f5eaf1fdc9f065f0007e91',
 '50f5eaf2fdc9f065f0007e99',
 '50f5eaf2fdc9f065f0007e9c',
 '50f5eaf2fdc9f065f0007e9d',
 '50f5eaf2fdc9f065f0007ea1',
 '50f5eaf2fdc9f065f0007ea2',
 '50f5eaf2fdc9f065f0007ea6',
 '50f5eaf3fdc9f065f0007eac',
 '50f5eaf3fdc9f065f0007eae',
 '50f5eaf3fdc9f065f0007eb6',
 '50f5eaf3fdc9f065f0007eb8',
 '50f5eaf3fdc9f065f0007ebe',
 '50f5eaf3fdc9f065f0007ec1',
 '50f5eaf4fdc9f065f0007ec5',
 '50f5eaf4fdc9f065f0007ec6',
 '50f5eaf4fdc9f065f0007ecd',
 '50f5eaf4fdc9f065f0007ed1',
 '50f5eaf4fdc9f065f0007ed3',
 '50f5eaf4fdc9f065f0007ed4',
 '50f5eaf6fdc9f065f0007efd',
 '50f5eaf9fdc9f065f0007f25',
 '50f5eb16fdc9f065f0007f48',
 '50f5eb23fdc9f065f000802e',
 '50f5eb25fdc9f065f0008060',
 '50f5eb26fdc9f065f0008074',
 '50f5eb27fdc9f065f0008083',
 '50f5eb43fdc9f065f0008140',
 '50f5eb5ffdc9f065f00081fe',
 '50f5eb5ffdc9f065f0008200',
 '50f5eb5ffdc9f065f0008203',
 '50f5eb60fdc9f065f000820c',
 '50f5eb60fdc9f065f0008211',
 '50f5eb60fdc9f065f0008216',
 '50f5eb60fdc9f065f0008219',
 '50f5eb61fdc9f065f0008224',
 '50f5eb61fdc9f065f000822b',
 '50f5eb62fdc9f065f0008237',
 '50f5eb62fdc9f065f0008244',
 '50f5eb63fdc9f065f0008245',
 '50f5eb63fdc9f065f0008248',
 '50f5eb63fdc9f065f000824f',
 '50f5eb63fdc9f065f0008253',
 '50f5eb63fdc9f065f0008258',
 '50f5eb64fdc9f065f0008265',
 '50f5eb64fdc9f065f0008267',
 '50f5eb64fdc9f065f0008268',
 '50f5eb64fdc9f065f000826c',
 '50f5eb64fdc9f065f000826d',
 '50f5eb65fdc9f065f0008273',
 '50f5eb66fdc9f065f0008286',
 '50f5eb66fdc9f065f000828d',
 '50f5eb66fdc9f065f000828e',
 '50f5eb67fdc9f065f0008294',
 '50f5eb67fdc9f065f0008298',
 '50f5eb67fdc9f065f000829f',
 '50f5eb68fdc9f065f00082a3',
 '50f5eb68fdc9f065f00082a7',
 '50f5eb68fdc9f065f00082ae',
 '50f5eb68fdc9f065f00082b0',
 '50f5eb69fdc9f065f00082bb',
 '50f5eb69fdc9f065f00082bc',
 '50f5eb69fdc9f065f00082c0',
 '50f5eb69fdc9f065f00082c9',
 '50f5eb69fdc9f065f00082cc',
 '50f5eb69fdc9f065f00082cd',
 '50f5eb6afdc9f065f00082d4',
 '50f5eb6afdc9f065f00082d5',
 '50f5eb6afdc9f065f00082d9',
 '50f5eb6afdc9f065f00082df',
 '50f5eb6afdc9f065f00082e1',
 '50f5eb6afdc9f065f00082e2',
 '50f5ebc6fdc9f065f00084ea',
 '50f5ebcafdc9f065f0008548',
 '50f5ebcbfdc9f065f000855e',
 '50f5ebd2fdc9f065f00085e6',
 '50f5ebd2fdc9f065f00085ed',
 '50f5ebd2fdc9f065f00085f0',
 '50f5ec0dfdc9f065f0008647',
 '50f5ec0efdc9f065f0008658',
 '50f5ec17fdc9f065f0008708',
 '50f5ec18fdc9f065f0008719',
 '50f5ec1ffdc9f065f00087a4',
 '50f5ec37fdc9f065f0008802',
 '50f5ec40fdc9f065f00088bc']

In [14]:
len(image_ids)

150

In [15]:
# Suponiendo que tu array de IDs ya está definido como image_ids
# Si no está definido, puedes copiar y pegar la definición de tu array

# Crea una lista de números del 1 al 150 (o el número de IDs que tengas)
contador = list(range(0, len(image_ids) ))

# Crea un DataFrame con las dos columnas
df = pd.DataFrame({
    'Contador': contador,
    'ID': image_ids
})


In [16]:
df.head()

,Contador,ID
0,0,50f5ea5bfdc9f065f0007ab6
1,1,50f5ea5bfdc9f065f0007ab9
2,2,50f5ea5cfdc9f065f0007abf
3,3,50f5ea5cfdc9f065f0007ac5
4,4,50f5ea5dfdc9f065f0007ada


In [17]:
df.shape

(150, 2)

In [18]:
df[ (df["Contador"] == 56 ) | (df["Contador"] == 117 ) ]

,Contador,ID
56,56,50f5eaf0fdc9f065f0007e7e
117,117,50f5eb66fdc9f065f000828e


In [ ]:
import shutil
import os
import pandas as pd

# Define las rutas

origen_dir = "/home/vdela/UrbaNet-main/data/pp2/images/Rio De Janeiro"
destino_dir = "/home/vdela/UrbaNet-main/data/pp2/Rio"
# Asegúrate de que la carpeta de destino exista
os.makedirs(destino_dir, exist_ok=True)


# Iterar y copiar
for _, row in df.iterrows():
    contador = row['Contador'] #- 1
    img_id = row['ID']
    
    archivo_origen = f"{img_id}.JPG"
    ruta_origen = os.path.join(origen_dir, archivo_origen)

    archivo_destino = f"{contador}.JPG"
    ruta_destino = os.path.join(destino_dir, archivo_destino)

    if os.path.exists(ruta_origen):
        shutil.copy2(ruta_origen, ruta_destino)
        print(f"Copiado {archivo_origen} → {archivo_destino}")
    else:
        print(f"No se encontró: {archivo_origen}")


Copiado 50f5ea5bfdc9f065f0007ab6.JPG → 0.JPG
Copiado 50f5ea5bfdc9f065f0007ab9.JPG → 1.JPG
Copiado 50f5ea5cfdc9f065f0007abf.JPG → 2.JPG
Copiado 50f5ea5cfdc9f065f0007ac5.JPG → 3.JPG
Copiado 50f5ea5dfdc9f065f0007ada.JPG → 4.JPG
Copiado 50f5ea5ffdc9f065f0007b06.JPG → 5.JPG
Copiado 50f5ea60fdc9f065f0007b19.JPG → 6.JPG
Copiado 50f5ea61fdc9f065f0007b1d.JPG → 7.JPG
Copiado 50f5ea61fdc9f065f0007b1f.JPG → 8.JPG
Copiado 50f5ea61fdc9f065f0007b22.JPG → 9.JPG
Copiado 50f5ea61fdc9f065f0007b27.JPG → 10.JPG
Copiado 50f5eaa5fdc9f065f0007b7b.JPG → 11.JPG
Copiado 50f5eaa5fdc9f065f0007b7c.JPG → 12.JPG
Copiado 50f5eaa5fdc9f065f0007b81.JPG → 13.JPG
Copiado 50f5eaa6fdc9f065f0007b8f.JPG → 14.JPG
Copiado 50f5eaa6fdc9f065f0007b94.JPG → 15.JPG
Copiado 50f5eaa7fdc9f065f0007ba0.JPG → 16.JPG
Copiado 50f5eaa7fdc9f065f0007ba4.JPG → 17.JPG
Copiado 50f5eaa9fdc9f065f0007bc7.JPG → 18.JPG
Copiado 50f5eac6fdc9f065f0007ca1.JPG → 19.JPG
Copiado 50f5eac6fdc9f065f0007cab.JPG → 20.JPG
Copiado 50f5eac7fdc9f065f0007cb2.JPG → 21.JP

In [34]:
from py.models.classification.cnn.vgg import VGG16

model = VGG16(num_classes=2, use_gap=True)
model.load_state_dict(torch.load(f"{MODEL_PATH}{cfg.MODEL_FEATURE_NAME}_best_model.pth"))
model.eval()

VGG16(
  (feature_maps): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dila

In [35]:
from py.models.explainers import GradCAM
from py.models.datasets.transformations import ImageTransforms
from PIL import Image

RIO_DIR    = f"{cfg.DATA_PATH}pp2/Rio"
GRADCAM_DIR = f"{RIO_DIR}/GradCAM"
COMP_DIR    = f"{RIO_DIR}/Comparations"

os.makedirs(GRADCAM_DIR, exist_ok=True)
os.makedirs(COMP_DIR,    exist_ok=True)

transforms_list = ImageTransforms().get(model_name=cfg.MODEL_FEATURE_NAME)
gradcam = GradCAM(model)

print(f"Directorios creados:")
print(f"  GradCAM     → {GRADCAM_DIR}")
print(f"  Comparisons → {COMP_DIR}")


Directorios creados:
  GradCAM     → /home/vdela/UrbaNet-main/data/pp2/Rio/GradCAM
  Comparisons → /home/vdela/UrbaNet-main/data/pp2/Rio/Comparations


In [36]:
errors = []

for i in tqdm(range(150), desc="GradCAM Rio"):
    img_path    = os.path.join(RIO_DIR, f"{i}.JPG")
    gc_path     = os.path.join(GRADCAM_DIR, f"{i}.JPG")
    comp_path   = os.path.join(COMP_DIR, f"{i}.JPG")

    if not os.path.exists(img_path):
        errors.append({"i": i, "error": "imagen no encontrada"})
        continue

    try:
        original = Image.open(img_path).convert("RGB")

        # --- GradCAM ---
        cam_heatmap, pred_class = gradcam.generate_cam(
            original,
            transforms_list=transforms_list["val"],
            target_class=None
        )
        viz, _ = gradcam.visualize(original, cam_heatmap)

        # Guardar solo el GradCAM
        Image.fromarray(viz).save(gc_path, quality=90)

        # --- Comparación: original | GradCAM ---
        orig_arr = np.array(original)
        w = orig_arr.shape[1]
        h = orig_arr.shape[0]
        viz_resized = np.array(Image.fromarray(viz).resize((w, h)))

        comparison = np.concatenate([orig_arr, viz_resized], axis=1)
        Image.fromarray(comparison).save(comp_path, quality=90)

    except Exception as e:
        errors.append({"i": i, "error": str(e)})

print(f"\nCompletado: {150 - len(errors)}/150 imágenes procesadas")
if errors:
    print(f"Errores ({len(errors)}):")
    for err in errors:
        print(f"  imagen {err['i']}: {err['error']}")


GradCAM Rio: 100%|██████████| 150/150 [00:15<00:00,  9.64it/s]


Completado: 150/150 imágenes procesadas


In [37]:
from PIL import Image
import numpy as np
import os
from tqdm import tqdm

RIO_DIR      = f"{cfg.DATA_PATH}pp2/Rio"
GRADCAM_DIR  = f"{RIO_DIR}/GradCAM"
HEATMAP_DIR  = f"{RIO_DIR}/heatmaps"
COMP2_DIR    = f"{RIO_DIR}/Comparations2"

os.makedirs(COMP2_DIR, exist_ok=True)

errors = []

for i in tqdm(range(150), desc="Comparations2"):
    orig_path    = os.path.join(RIO_DIR,     f"{i}.JPG")
    gc_path      = os.path.join(GRADCAM_DIR, f"{i}.JPG")
    heatmap_path = os.path.join(HEATMAP_DIR, f"heatmap_{i}.png")
    out_path     = os.path.join(COMP2_DIR,   f"{i}.JPG")

    missing = [p for p in [orig_path, gc_path, heatmap_path] if not os.path.exists(p)]
    if missing:
        errors.append({"i": i, "missing": missing})
        continue

    try:
        orig    = Image.open(orig_path).convert("RGB")
        gc      = Image.open(gc_path).convert("RGB")
        heatmap = Image.open(heatmap_path).convert("RGB")

        # Usar el tamaño del original como referencia
        w, h = orig.size
        gc      = gc.resize((w, h), Image.LANCZOS)
        heatmap = heatmap.resize((w, h), Image.LANCZOS)

        # Concatenar horizontalmente: original | GradCAM | heatmap
        comparison = Image.new("RGB", (w * 3, h))
        comparison.paste(orig,    (0,     0))
        comparison.paste(gc,      (w,     0))
        comparison.paste(heatmap, (w * 2, 0))

        comparison.save(out_path, quality=90)

    except Exception as e:
        errors.append({"i": i, "error": str(e)})

print(f"\nCompletado: {150 - len(errors)}/150 imágenes procesadas")
print(f"Guardado en: {COMP2_DIR}")
if errors:
    print(f"\nErrores ({len(errors)}):")
    for err in errors:
        print(f"  imagen {err['i']}: {err}")


Comparations2: 100%|██████████| 150/150 [00:01<00:00, 118.62it/s]


Completado: 150/150 imágenes procesadas
Guardado en: /home/vdela/UrbaNet-main/data/pp2/Rio/Comparations2


In [38]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import os
from tqdm import tqdm

RIO_DIR      = f"{cfg.DATA_PATH}pp2/Rio"
GRADCAM_DIR  = f"{RIO_DIR}/GradCAM"
HEATMAP_DIR  = f"{RIO_DIR}/heatmaps"
COMP2_DIR    = f"{RIO_DIR}/Comparations2"

os.makedirs(COMP2_DIR, exist_ok=True)

LABELS      = ["Original", "GradCAM", "EyeTracking"]
LABEL_H     = 40           # altura de la barra de etiquetas en px
BG_COLOR    = (30, 30, 30) # fondo oscuro
TEXT_COLOR  = (255, 255, 255)

# Intentar cargar fuente del sistema; si no, usar la default
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
except:
    font = ImageFont.load_default()

def add_label_bar(panel: Image.Image, label: str) -> Image.Image:
    """Agrega una barra de etiqueta encima de un panel."""
    w, h = panel.size
    bar = Image.new("RGB", (w, LABEL_H), BG_COLOR)
    draw = ImageDraw.Draw(bar)
    bbox = draw.textbbox((0, 0), label, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    draw.text(((w - text_w) // 2, (LABEL_H - text_h) // 2), label, fill=TEXT_COLOR, font=font)
    combined = Image.new("RGB", (w, LABEL_H + h))
    combined.paste(bar,   (0, 0))
    combined.paste(panel, (0, LABEL_H))
    return combined

errors = []

for i in tqdm(range(150), desc="Comparations2"):
    orig_path    = os.path.join(RIO_DIR,     f"{i}.JPG")
    gc_path      = os.path.join(GRADCAM_DIR, f"{i}.JPG")
    heatmap_path = os.path.join(HEATMAP_DIR, f"heatmap_{i}.png")
    out_path     = os.path.join(COMP2_DIR,   f"{i}.JPG")

    missing = [p for p in [orig_path, gc_path, heatmap_path] if not os.path.exists(p)]
    if missing:
        errors.append({"i": i, "missing": missing})
        continue

    try:
        orig    = Image.open(orig_path).convert("RGB")
        gc      = Image.open(gc_path).convert("RGB")
        heatmap = Image.open(heatmap_path).convert("RGB")

        w, h = orig.size
        gc      = gc.resize((w, h), Image.LANCZOS)
        heatmap = heatmap.resize((w, h), Image.LANCZOS)

        # Añadir etiqueta a cada panel
        panel_orig    = add_label_bar(orig,    LABELS[0])
        panel_gc      = add_label_bar(gc,      LABELS[1])
        panel_heatmap = add_label_bar(heatmap, LABELS[2])

        # Concatenar horizontalmente
        total_w = w * 3
        total_h = LABEL_H + h
        comparison = Image.new("RGB", (total_w, total_h))
        comparison.paste(panel_orig,    (0,     0))
        comparison.paste(panel_gc,      (w,     0))
        comparison.paste(panel_heatmap, (w * 2, 0))

        comparison.save(out_path, quality=90)

    except Exception as e:
        errors.append({"i": i, "error": str(e)})

print(f"\nCompletado: {150 - len(errors)}/150 imágenes procesadas")
print(f"Guardado en: {COMP2_DIR}")
if errors:
    print(f"\nErrores ({len(errors)}):")
    for err in errors:
        print(f"  imagen {err['i']}: {err}")

Comparations2: 100%|██████████| 150/150 [00:01<00:00, 125.46it/s]


Completado: 150/150 imágenes procesadas
Guardado en: /home/vdela/UrbaNet-main/data/pp2/Rio/Comparations2
